In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from geoai.utils_ds.dataframe_ops import DataFrameOperations
from geoai.utils_ds.preprocessing_ops import PreProcessingOperations
from geoai.utils_ds.visualize_ops import VisualizeOperations
from geoai.utils_ml.model_ops import ModelOperations

df_ops = DataFrameOperations()
vis_ops = VisualizeOperations()
pre_ops = PreProcessingOperations()
model_ops = ModelOperations()

In [2]:
# Load the data
X_train = pd.read_csv("csv_files/X_train_with_indices.csv")
X_test = pd.read_csv("csv_files/X_test_with_indices.csv")
y_train = pd.read_csv("csv_files/y_train_encoded.csv")
y_test = pd.read_csv("csv_files/y_test_encoded.csv")
X_train.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI
0,350.0,542.0000,323.0,3277.0000,1975.3334,0.820556,-0.247826,0.002545
1,390.0,555.0000,380.0,3016.6667,1991.0000,0.776251,-0.204819,0.002227
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000,0.044101,0.098425,0.000127
3,363.2,546.5000,395.0,3244.5000,2052.0000,0.782937,-0.225149,0.002438
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000,0.031066,0.123170,0.000098


# Scale the data


Scaling data is a crucial preprocessing step in machine learning that can significantly impact the performance of algorithms. Here are some reasons why scaling is essential:

### 1. **Improves Model Performance**

Many machine learning algorithms, such as logistic regression rely on the distances between data points. If the features are on different scales, these distances can become distorted, leading to suboptimal performance. Scaling ensures that each feature contributes equally to the result.

### 2. **Speeds Up Convergence**

Gradient-based optimization algorithms, such as gradient descent used in neural networks and linear regression, converge faster when the data is scaled. When features are on similar scales, the algorithm takes more efficient steps towards the minimum of the cost function, speeding up the learning process.

### 3. **Ensures Consistent Feature Contribution**

Without scaling, features with larger ranges can dominate the learning process, leading the model to give them undue importance. Scaling ensures that each feature is given equal weight, improving the overall accuracy and reliability of the model.

### Min-Max Scaling

One common method of scaling is min-max scaling, also known as normalization.
This technique transforms the features to a fixed range, typically [0, 1].

### Benefits of Min-Max Scaling

- **Preserves Relationships**: Min-max scaling preserves the relationships between values, ensuring that the transformed data maintains the same structure as the original data.
- **Feature Range Consistency**: By bringing all features into a similar range, it ensures that no single feature will dominate others, allowing for more balanced and fair modeling.

In [5]:
# scale the data
X_train_scaled, X_test_scaled = pre_ops.scale_to_minmax(X_train, X_test)

In [6]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=1)
lr.fit(X_train_scaled, y_train.values.ravel())

# predict the validation set
y_train_pred = lr.predict(X_train_scaled)
y_pred_test = lr.predict(X_test_scaled)

In [8]:
# calculate the accuracy
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)[3]}")
print(f"Test Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_pred_test)[3]}")

Train Accuracy: 0.9103285032025857
Test Accuracy: 0.9037308016505567


# Use the data with Engineered Features

## Importance of Having Many Features in a Model

Having many features in a machine learning model may improve performance of a
model. The more relevant features a model has, the more information it can
leverage to learn patterns and make accurate predictions.

### Why Many Features Are Important

1. **Captures Complexity**: Real-world data often have complex relationships and interactions. Having many features allows the model to capture these nuances, leading to more accurate predictions.
2. **Improves Predictive Power**: More features provide the model with more information, which can enhance its ability to distinguish between different classes or predict continuous outcomes accurately.
3. **Incorporates Diverse Data**: Including various features ensures that different perspectives and dimensions of the data are considered, making the model more robust and versatile.

### Pros of Having Many Features

1. **Enhanced Accuracy**: More features may lead to better model performance by providing additional information that helps the model learn more effectively.
2. **Rich Insights**: A model with many features can offer deeper insights into the relationships and interactions within the data, aiding in better decision-making.
3. **Improved Generalization**: With a diverse set of features, the model is more likely to generalize well to new, unseen data, as it can capture a broader range of patterns.

### Cons of Having Many Features

1. **Increased Complexity**: More features can make the model more complex, leading to longer training times and higher computational costs.
2. **Risk of Overfitting**: With too many features, the model might overfit the training data, capturing noise instead of the underlying patterns, resulting in poor performance on new data.
3. **Feature Redundancy**: Some features may be redundant or irrelevant, adding noise to the model and potentially degrading its performance.
4. **Interpretability Issues**: As the number of features increases, the model becomes harder to interpret, making it challenging to understand the contribution of each feature to the predictions.

In [6]:
X_train_fe = pd.read_csv("csv_files/X_train_fe.csv")
X_test_fe = pd.read_csv("csv_files/X_test_fe.csv")
X_train_fe.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_binary_non_veg,NDVI_binary_veg,NDVI_categorized_encoded
0,5.860786,6.297109,5.780744,8.094989,7.588999,0.599142,-0.284788,0.002541,34.348815,36.906012,...,0.019287,0.358971,-0.170628,0.001523,0.081104,-0.000724,6.459135e-06,0,1,2
1,5.968708,6.320768,5.942799,8.012239,7.596894,0.574505,-0.229186,0.002224,35.625470,37.726818,...,0.016899,0.330056,-0.131668,0.001278,0.052526,-0.000510,4.948067e-06,0,1,2
2,7.071573,7.111240,7.145984,7.234177,7.431537,0.043156,0.093877,0.000127,50.007150,50.287656,...,0.000947,0.001862,0.004051,0.000006,0.008813,0.000012,1.625104e-08,1,0,0
3,5.897703,6.305362,5.981414,8.085025,7.627057,0.578262,-0.255084,0.002435,34.782903,37.187156,...,0.018575,0.334387,-0.147505,0.001408,0.065068,-0.000621,5.931214e-06,0,1,2
4,6.999970,7.010613,7.051856,7.113956,7.361375,0.030594,0.116155,0.000098,48.999577,49.074077,...,0.000724,0.000936,0.003554,0.000003,0.013492,0.000011,9.665850e-09,1,0,0


In [7]:
# scale the data
X_train_fe_scaled, X_test_fe_scaled = pre_ops.scale_to_minmax(X_train_fe, X_test_fe)
X_train_fe_scaled.head()

,BLUE,GREEN,RED,NIR,SWIR,NDVI,NDBI,REI,BLUE^2,BLUE GREEN,...,SWIR REI,NDVI^2,NDVI NDBI,NDVI REI,NDBI^2,NDBI REI,REI^2,NDVI_binary_non_veg,NDVI_binary_veg,NDVI_categorized_encoded
0,0.028792,0.092959,0.036666,0.811104,0.252389,0.998145,0.074872,0.958466,0.023717,0.048942,...,0.967258,0.996101,0.115180,0.975468,0.741622,0.087732,0.918597,0.0,1.0,1.0
1,0.070846,0.103193,0.096423,0.752344,0.257980,0.959140,0.165553,0.838990,0.058899,0.072450,...,0.847577,0.915867,0.281023,0.819191,0.480302,0.279052,0.703698,0.0,1.0,1.0
2,0.500597,0.445141,0.540090,0.199853,0.140895,0.117894,0.692431,0.048801,0.455225,0.432197,...,0.048329,0.005168,0.858749,0.006760,0.080586,0.745577,0.002311,1.0,0.0,0.0
3,0.043177,0.096529,0.110662,0.804029,0.279338,0.965088,0.123316,0.918495,0.035680,0.056994,...,0.931568,0.927885,0.213608,0.902424,0.594985,0.179424,0.843518,0.0,1.0,1.0
4,0.472696,0.401611,0.505381,0.114486,0.091216,0.098004,0.728764,0.037811,0.427459,0.397440,...,0.037124,0.002597,0.856630,0.005167,0.123372,0.745088,0.001375,1.0,0.0,0.0


In [14]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr.fit(X_train_fe_scaled, y_train.values.ravel())

# predict the validation set
y_train_pred_fe = lr.predict(X_train_fe_scaled)
y_pred_test_fe = lr.predict(X_test_fe_scaled)

In [15]:
# calculate the accuracy
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred_fe)[3]}")
print(f"Test Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_pred_test_fe)[3]}")


Train Accuracy: 0.9205994488039595
Test Accuracy: 0.9188361506180447


END